In [1]:
BUILD_KIND = "ocr"

MAPPING_HINT = "aic-mapping"
CODE_HINT = "update-script"
RAW_HINT = "ocr"       # sửa theo tên Dataset OCR thực tế
CONFIG_HINT = None

In [2]:
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

dataset_roots = []

for category in ("datasets", "models", "notebooks"):
    category_root = INPUT_ROOT / category

    if not category_root.exists():
        continue

    for owner_root in category_root.iterdir():
        if not owner_root.is_dir():
            continue

        for dataset_root in owner_root.iterdir():
            if dataset_root.is_dir():
                dataset_roots.append(dataset_root)

print("Các input đang được gắn:")

for root in dataset_roots:
    print("-", root)

Các input đang được gắn:
- /kaggle/input/datasets/ltnngoc/aic2026-ocr
- /kaggle/input/datasets/blonton/aic-mapping
- /kaggle/input/models/blonton/update-script-v2


In [3]:
from pathlib import Path


def find_input_root(hint, required=True):
    if not hint:
        return None

    hint = hint.casefold()

    matches = [
        path
        for path in dataset_roots
        if hint in path.name.casefold()
        or hint in str(path).casefold()
    ]

    if not matches:
        if required:
            raise FileNotFoundError(
                f"Không tìm thấy Kaggle input chứa tên: {hint}"
            )
        return None

    if len(matches) > 1:
        print(f"Có nhiều kết quả cho {hint!r}:")
        for path in matches:
            print(" -", path)

    return matches[0]


MAPPING_SOURCE_ROOT = find_input_root(MAPPING_HINT)
CODE_SOURCE_ROOT = find_input_root(CODE_HINT)
RAW_SOURCE_ROOT = find_input_root(RAW_HINT)

CONFIG_SOURCE_ROOT = find_input_root(
    CONFIG_HINT,
    required=False,
)

print("MAPPING:", MAPPING_SOURCE_ROOT)
print("CODE   :", CODE_SOURCE_ROOT)
print("RAW    :", RAW_SOURCE_ROOT)
print("CONFIG :", CONFIG_SOURCE_ROOT)

MAPPING: /kaggle/input/datasets/blonton/aic-mapping
CODE   : /kaggle/input/models/blonton/update-script-v2
RAW    : /kaggle/input/datasets/ltnngoc/aic2026-ocr
CONFIG : None


In [4]:
import os
from pathlib import Path


def find_named_file(root, filename, max_depth=8):
    root = Path(root)
    root_depth = len(root.parts)

    for current, directories, files in os.walk(root):
        current_path = Path(current)
        depth = len(current_path.parts) - root_depth

        if depth >= max_depth:
            directories[:] = []

        if filename in files:
            return current_path / filename

    return None


required_build_script = {
    "object": "build_object_index.py",
    "ocr": "build_ocr_index.py",
    "asr": "build_asr_index.py",
}[BUILD_KIND]

BUILD_SCRIPT_SOURCE = find_named_file(
    CODE_SOURCE_ROOT,
    required_build_script,
)

MAPPING_SOURCE = find_named_file(
    MAPPING_SOURCE_ROOT,
    "clip_row_mapping.jsonl",
)

assert BUILD_SCRIPT_SOURCE, (
    f"Không tìm thấy {required_build_script} trong code input"
)

assert MAPPING_SOURCE, (
    "Không tìm thấy clip_row_mapping.jsonl trong aic-mapping"
)

CODE_SCRIPTS_SOURCE = BUILD_SCRIPT_SOURCE.parent

print("Thư mục scripts:", CODE_SCRIPTS_SOURCE)
print("Mapping:", MAPPING_SOURCE)

Thư mục scripts: /kaggle/input/models/blonton/update-script-v2/pytorch/default/1
Mapping: /kaggle/input/datasets/blonton/aic-mapping/artifacts/clip_row_mapping.jsonl


In [5]:
import shutil
from pathlib import Path

PROJECT_ROOT = Path("/kaggle/working/aic_practice")
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
DATA_DIR = PROJECT_ROOT / "data"
CONFIG_DIR = PROJECT_ROOT / "config"

for directory in (
    PROJECT_ROOT,
    SCRIPTS_DIR,
    ARTIFACTS_DIR,
    DATA_DIR,
    CONFIG_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)

Project: /kaggle/working/aic_practice


In [6]:
copied_files = []

for source in CODE_SCRIPTS_SOURCE.glob("*.py"):
    destination = SCRIPTS_DIR / source.name
    shutil.copy2(source, destination)
    copied_files.append(source.name)

assert required_build_script in copied_files, (
    f"Chưa copy được {required_build_script}"
)

print("Số file code đã copy:", len(copied_files))
print("Build script:", SCRIPTS_DIR / required_build_script)

Số file code đã copy: 33
Build script: /kaggle/working/aic_practice/scripts/build_ocr_index.py


In [7]:
MAPPING_TARGET = ARTIFACTS_DIR / "clip_row_mapping.jsonl"

shutil.copy2(
    MAPPING_SOURCE,
    MAPPING_TARGET,
)

print("Mapping:", MAPPING_TARGET)
print(
    "Dung lượng:",
    round(MAPPING_TARGET.stat().st_size / 1024**2, 2),
    "MB",
)

Mapping: /kaggle/working/aic_practice/artifacts/clip_row_mapping.jsonl
Dung lượng: 109.47 MB


In [8]:
DATA_TARGET = DATA_DIR / {
    "object": "objects",
    "ocr": "ocr",
    "asr": "asr",
}[BUILD_KIND]

if DATA_TARGET.is_symlink():
    DATA_TARGET.unlink()
elif DATA_TARGET.exists():
    if DATA_TARGET.is_dir() and not any(DATA_TARGET.iterdir()):
        DATA_TARGET.rmdir()
    else:
        raise RuntimeError(
            f"{DATA_TARGET} đã tồn tại và không rỗng. "
            "Hãy kiểm tra trước khi thay thế."
        )

DATA_TARGET.symlink_to(
    RAW_SOURCE_ROOT,
    target_is_directory=True,
)

print("Nguồn:", RAW_SOURCE_ROOT)
print("Đã gắn vào:", DATA_TARGET)
print("Symlink:", DATA_TARGET.is_symlink())

Nguồn: /kaggle/input/datasets/ltnngoc/aic2026-ocr
Đã gắn vào: /kaggle/working/aic_practice/data/ocr
Symlink: True


In [9]:
print("PROJECT_ROOT:", PROJECT_ROOT)
print("BUILD_KIND:", BUILD_KIND)
print("Build script tồn tại:", (SCRIPTS_DIR / required_build_script).is_file())
print("Mapping tồn tại:", MAPPING_TARGET.is_file())
print("Data target tồn tại:", DATA_TARGET.exists())
print("Data target là symlink:", DATA_TARGET.is_symlink())

sample_files = []

for suffix in ("*.json", "*.jsonl"):
    for path in DATA_TARGET.rglob(suffix):
        sample_files.append(path)

        if len(sample_files) >= 10:
            break

    if len(sample_files) >= 10:
        break

print("\nFile dữ liệu mẫu:")

for path in sample_files:
    print("-", path)

assert sample_files, (
    f"Không tìm thấy JSON/JSONL trong {DATA_TARGET}"
)

PROJECT_ROOT: /kaggle/working/aic_practice
BUILD_KIND: ocr
Build script tồn tại: True
Mapping tồn tại: True
Data target tồn tại: True
Data target là symlink: True

File dữ liệu mẫu:
- /kaggle/working/aic_practice/data/ocr/ocr_results_part1/L21_V017/014500.json
- /kaggle/working/aic_practice/data/ocr/ocr_results_part1/L21_V017/003900.json
- /kaggle/working/aic_practice/data/ocr/ocr_results_part1/L21_V017/006275.json
- /kaggle/working/aic_practice/data/ocr/ocr_results_part1/L21_V017/003225.json
- /kaggle/working/aic_practice/data/ocr/ocr_results_part1/L21_V017/006000.json
- /kaggle/working/aic_practice/data/ocr/ocr_results_part1/L21_V017/001350.json
- /kaggle/working/aic_practice/data/ocr/ocr_results_part1/L21_V017/016750.json
- /kaggle/working/aic_practice/data/ocr/ocr_results_part1/L21_V017/010150.json
- /kaggle/working/aic_practice/data/ocr/ocr_results_part1/L21_V017/022625.json
- /kaggle/working/aic_practice/data/ocr/ocr_results_part1/L21_V017/021700.json


In [10]:
%cd /kaggle/working/aic_practice
!python -u scripts/build_ocr_index.py \
    --input data/ocr \
    --output artifacts/ocr_index.sqlite3

/kaggle/working/aic_practice
Đã index 10,000 OCR keyframe...
Đã index 20,000 OCR keyframe...
Đã index 30,000 OCR keyframe...
Đã index 40,000 OCR keyframe...
Đã index 50,000 OCR keyframe...
Đã index 60,000 OCR keyframe...
Đã index 70,000 OCR keyframe...
Đã index 80,000 OCR keyframe...
Đã index 90,000 OCR keyframe...
Đã index 100,000 OCR keyframe...
Đã index 110,000 OCR keyframe...
Đã index 120,000 OCR keyframe...
Đã index 130,000 OCR keyframe...
Đã index 140,000 OCR keyframe...
Đã index 150,000 OCR keyframe...
Đã index 160,000 OCR keyframe...
Đã index 170,000 OCR keyframe...
Đã index 180,000 OCR keyframe...
Đã index 190,000 OCR keyframe...
Đã index 200,000 OCR keyframe...
Đã index 210,000 OCR keyframe...
Tạo OCR index thành công: artifacts/ocr_index.sqlite3
  Indexed: 213,986; rỗng: 15,026; trùng: 0; không map được: 0


In [11]:
from pathlib import Path

expected_index = {
    "object": PROJECT_ROOT / "artifacts/object_index.sqlite3",
    "ocr": PROJECT_ROOT / "artifacts/ocr_index.sqlite3",
    "asr": PROJECT_ROOT / "artifacts/asr_index.sqlite3",
}[BUILD_KIND]

assert expected_index.is_file(), (
    f"Build chưa tạo được {expected_index}"
)

print("✅ Build thành công:", expected_index)
print(
    "Dung lượng:",
    round(expected_index.stat().st_size / 1024**2, 2),
    "MB",
)

✅ Build thành công: /kaggle/working/aic_practice/artifacts/ocr_index.sqlite3
Dung lượng: 338.87 MB


In [12]:
import shutil
from pathlib import Path

EXPORT_DIR = Path(
    f"/kaggle/working/aic2026_{BUILD_KIND}_index"
)

EXPORT_ARTIFACTS = EXPORT_DIR / "artifacts"
EXPORT_ARTIFACTS.mkdir(
    parents=True,
    exist_ok=True,
)

shutil.copy2(
    expected_index,
    EXPORT_ARTIFACTS / expected_index.name,
)

shutil.copy2(
    MAPPING_TARGET,
    EXPORT_ARTIFACTS / "clip_row_mapping.jsonl",
)

archive_path = shutil.make_archive(
    str(EXPORT_DIR),
    "zip",
    root_dir=EXPORT_DIR,
)

print("✅ Thư mục output:", EXPORT_DIR)
print("✅ File ZIP:", archive_path)

✅ Thư mục output: /kaggle/working/aic2026_ocr_index
✅ File ZIP: /kaggle/working/aic2026_ocr_index.zip
